In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import polars as pl
from dotenv import load_dotenv

from fr_gov_api import PrimClient


In [3]:
p_conf = "/home/valentin/work/conf/ratp.env"
load_dotenv(p_conf)

True

In [4]:
# Get perimeter data from PRIM website - pqt format
# https://prim.iledefrance-mobilites.fr/fr/jeux-de-donnees/perimetre-des-donnees-tr-disponibles-plateforme-idfm

In [5]:
p_pqt = "datasets/perimetre-des-donnees-tr-disponibles-plateforme-idfm.parquet"
df = pl.read_parquet(p_pqt)

In [6]:
dtype = pl.Struct(
    {
        "@srsName": pl.String,
        "ns2:Longitude": pl.String,
        "ns2:Latitude": pl.String,
    }
)
df = df.with_columns(pl.col("ns2_location").str.json_decode(dtype))


In [7]:
dtype = pl.Struct({"ns2:LineRef": pl.List(pl.String)})
df = df.with_columns(pl.col("ns2_lines").str.json_decode(dtype))

In [8]:
df = df.unnest("ns2_lines", "ns2_location")

In [9]:
df.head()

line,name_line,ns2_stoppointref,ns2_stopname,ns2:LineRef,@srsName,ns2:Longitude,ns2:Latitude,ns2_location_ns2longitude,operatorname
str,str,str,str,list[str],str,str,str,str,str
"""STIF:Line::C02107:""","""3137""","""STIF:StopPoint:Q:6215:""","""Ãglise""","[""STIF:Line::C02107:"", ""STIF:Line::C02108:"", … ""STIF:Line::C02088:""]","""EPSG:2154""","""6841919.0""","""681371.0""","""6841919.0""","""Keolis Portes et Val de Brie"""
"""STIF:Line::C00115:""","""6593""","""STIF:StopPoint:Q:478867:""",null,"[""STIF:Line::C00111:"", ""STIF:Line::C00115:""]",null,"""0.0""","""0.0""","""0.0""","""Keolis Seine et Oise Est"""
"""STIF:Line::C02470:""","""4411""","""STIF:StopPoint:Q:2759:""","""Gare d'Ãtampes""","[""STIF:Line::C01851:"", ""STIF:Line::C01568:"", … ""STIF:Line::C00982:""]","""EPSG:2154""","""6815469.0""","""637847.0""","""6815469.0""","""Francilité Ouest Essonne"""
"""STIF:Line::C01224:""","""210""","""STIF:StopPoint:Q:479089:""","""Rue du 4eme Zouave""","[""STIF:Line::C01224:""]","""EPSG:2154""","""6859788.0""","""664990.0""","""6859788.0""","""RATP Cap Boucles de Marne"""
"""STIF:Line::C01902:""","""F""","""STIF:StopPoint:Q:27164:""","""La Fontaine""","[""STIF:Line::C01902:""]","""EPSG:2154""","""6845732.0""","""646200.0""","""6845732.0""","""Autocars Dominique"""


In [10]:
dfi = df.filter(
    pl.col("name_line") == "L",
    pl.col("ns2_stopname") == "Saint-Cloud"
)

In [11]:
dfi

line,name_line,ns2_stoppointref,ns2_stopname,ns2:LineRef,@srsName,ns2:Longitude,ns2:Latitude,ns2_location_ns2longitude,operatorname
str,str,str,str,list[str],str,str,str,str,str
"""STIF:Line::C01740:""","""L""","""STIF:StopPoint:Q:471421:""","""Saint-Cloud""","[""STIF:Line::C01741:"", ""STIF:Line::C01736:"", ""STIF:Line::C01740:""]","""EPSG:2154""","""6860963.0""","""642589.0""","""6860963.0""","""SNCF"""
"""STIF:Line::C01740:""","""L""","""STIF:StopPoint:Q:471906:""","""Saint-Cloud""","[""STIF:Line::C01741:"", ""STIF:Line::C01736:"", ""STIF:Line::C01740:""]","""EPSG:2154""","""6860960.0""","""642578.0""","""6860960.0""","""SNCF"""
"""STIF:Line::C01740:""","""L""","""STIF:StopPoint:Q:471432:""","""Saint-Cloud""","[""STIF:Line::C01741:"", ""STIF:Line::C01736:"", ""STIF:Line::C01740:""]","""EPSG:2154""","""6860962.0""","""642574.0""","""6860962.0""","""SNCF"""
"""STIF:Line::C01740:""","""L""","""STIF:StopPoint:Q:470852:""","""Saint-Cloud""","[""STIF:Line::C01741:"", ""STIF:Line::C01736:"", ""STIF:Line::C01740:""]","""EPSG:2154""","""6860962.0""","""642594.0""","""6860962.0""","""SNCF"""


In [12]:
ligne_ref = dfi.get_column("line").to_list()[0]
ligne_ref

'STIF:Line::C01740:'

In [24]:
prim = PrimClient()
data = prim.estimated_timetable()

In [26]:
df = pl.DataFrame(data)

In [27]:
df

RecordedAtTime,LineRef,DirectionRef,DatedVehicleJourneyRef,VehicleMode,VehicleFeatureRef,RouteRef,PublishedLineName,DirectionName,OriginRef,OriginName,DestinationRef,DestinationName,DestinationShortName,OperatorRef,ProductCategoryRef,JourneyNote,EstimatedCalls,VehicleJourneyName,FirstOrLastJourney
str,struct[1],struct[1],struct[1],list[str],list[str],struct[1],list[struct[1]],list[struct[1]],struct[1],list[struct[1]],struct[1],list[struct[1]],list[struct[1]],struct[1],struct[0],list[struct[1]],struct[1],list[struct[1]],struct[1]
"""2026-06-28T19:06:41.445Z""","{""STIF:Line::C01304:""}","{""Aller""}","{""MeC_Bus_PC:VehicleJourney::20260628.27.A.C01304:LOC""}","[""BUS""]",[],{null},"[{""360""}]","[{""Hôpital de Garches""}]",{null},[],"{""STIF:StopPoint:Q:490832:""}","[{""Hôpital de Garches""}]","[{""Hopital Garches""}]","{""MeC_Bus_PC:Operator::1055:""}",{},[],"{[{{""STIF:StopPoint:Q:29263:""},""2026-06-28T19:12:03.000Z"",[{""Hopital Garches""}],null,null,null,null,""ON_TIME"",null,null,null,null,null,null}]}",[],null
"""2026-06-28T01:00:22.122Z""","{""STIF:Line::C01852:""}","{""Aller""}","{""SNCF_MAGENTA_PRD:VehicleJourney::498645d2-5659-4fc2-9f84-c9bc921400a0:LOC""}","[""BUS""]","[""longTrain""]",{null},"[{""D Bus""}]",[],"{""STIF:StopPoint:Q:471564:""}","[{""JUVISY GARE ROUTIERE MAIRIE""}]","{""STIF:StopPoint:Q:40031:""}","[{""PARIS GARE DE LYON – Rue de Bercy""}]",null,{null},{},[],"{[{{""STIF:StopPoint:Q:17404:""},""2026-06-28T21:10:00.000Z"",[{""Paris Gare de Lyon""}],null,null,{""VILLENEUVE SAINT GEORGES""},{""VILLENEUVE SAINT GEORGES""},""ON_TIME"",""2026-06-28T21:10:00.000Z"",""2026-06-28T21:10:00.000Z"",""ON_TIME"",""2026-06-28T21:10:00.000Z"",null,null}, {{""STIF:StopPoint:Q:471564:""},""2026-06-28T20:40:00.000Z"",[{""Paris Gare de Lyon""}],null,null,{""JUVISY GARE ROUTIERE MAIRIE""},{""JUVISY GARE ROUTIERE MAIRIE""},""ON_TIME"",null,null,""ON_TIME"",""2026-06-28T20:40:00.000Z"",null,null}, … {{""STIF:StopPoint:Q:471352:""},""2026-06-28T21:01:00.000Z"",[{""Paris Gare de Lyon""}],null,null,{""VIGNEUX SUR SEINE""},{""VIGNEUX SUR SEINE""},""ON_TIME"",""2026-06-28T21:01:00.000Z"",""2026-06-28T21:01:00.000Z"",""ON_TIME"",""2026-06-28T21:01:00.000Z"",null,null}]}","[{""93630717""}]",null
"""2026-06-28T19:09:32.547Z""","{""STIF:Line::C00281:""}",null,"{""034_KSOE:VehicleJourney::5686553:LOC""}","[""BUS""]",[],{null},"[{""6503""}]","[{""Retour""}]",{null},[],"{""STIF:StopPoint:Q:427039:""}","[{""Saint-Exupéry""}]",null,"{""034_KSOE:Operator::KSOE:""}",{},"[{""L6503P18""}]","{[{{""STIF:StopPoint:Q:427039:""},null,[],null,null,null,null,null,""2026-06-28T21:07:00.000Z"",""2026-06-28T21:07:00.000Z"",""ON_TIME"",null,null,null}, {{""STIF:StopPoint:Q:13973:""},""2026-06-28T21:01:24.000Z"",[],null,null,null,null,""ON_TIME"",""2026-06-28T21:01:24.000Z"",""2026-06-28T21:01:24.000Z"",""ON_TIME"",""2026-06-28T21:01:24.000Z"",null,null}, … {{""STIF:StopPoint:Q:427043:""},""2026-06-28T20:58:00.000Z"",[],null,null,null,null,""ON_TIME"",""2026-06-28T20:58:00.000Z"",""2026-06-28T20:58:00.000Z"",""ON_TIME"",""2026-06-28T20:58:00.000Z"",null,null}]}",[],null
"""2026-06-28T18:11:13.766Z""","{""STIF:Line::C01737:""}","{""Aller""}","{""SNCF_MAGENTA_PRD:VehicleJourney::24630888-b780-4742-b67d-1db839d89713:LOC""}","[""RAIL""]","[""shortTrain""]",{null},"[{""H""}]",[],"{""STIF:StopArea:SP:43178:""}","[{""Persan - Beaumont""}]","{""STIF:StopArea:SP:462394:""}","[{""Paris Gare du Nord""}]",null,{null},{},"[{""AVOP""}]","{[{{""STIF:StopArea:SP:47582:""},""2026-06-28T20:23:10.000Z"",[{""Paris Gare du Nord""}],{{""STIF:StopPoint:Q:472612:""}},{{""STIF:StopPoint:Q:472612:""}},{""2""},{""2""},""ON_TIME"",""2026-06-28T20:22:30.000Z"",""2026-06-28T20:22:30.000Z"",""ON_TIME"",""2026-06-28T20:23:10.000Z"",null,null}, {{""STIF:StopArea:SP:47963:""},""2026-06-28T20:32:40.000Z"",[{""Paris Gare du Nord""}],{{""STIF:StopPoint:Q:472910:""}},{{""STIF:StopPoint:Q:472910:""}},{""2""},{""2""},""ON_TIME"",""2026-06-28T20:32:00.000Z"",""2026-06-28T20:32:00.000Z"",""ON_TI